In [29]:
import numpy as np
from astropy.table import Table
from astropy.table import join
from astropy.table import vstack
from astropy.io import fits

from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN
import networkx as nx

import readfof
from collections import Counter

from astropy.io import fits
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# plt.style.use('dark_background')
# import seaborn as sns

In [2]:
snapdir = '/pscratch/sd/v/vtorresg/quijotes/Halos/FoF/fiducial/0'
snapnum = 3

FoF = readfof.FoF_catalog(snapdir, snapnum, long_ids=False,
                          swap=False, SFR=False, read_IDs=False)
pos = FoF.GroupPos/1e3
pos.shape, type(pos)

((309740, 3), numpy.ndarray)

In [3]:
randv_all = Table.read('/pscratch/sd/v/vtorresg/quijotes/ASTRA/FoF/fiducial/0/group_003_random_voids.fits.gz').to_pandas()
randv_all.shape

(13077247, 5)

In [4]:
randv = randv_all[randv_all['RANDITER']==0]
randv.shape

(130670, 5)

In [5]:
pos_rand = randv[['X', 'Y', 'Z']].to_numpy()
pos_rand.shape

(130670, 3)

In [6]:
prob = Table.read('/pscratch/sd/v/vtorresg/quijotes/ASTRA/FoF/fiducial/0/group_003_probability.fits.gz').to_pandas()
prob.shape

(309740, 4)

In [7]:
assert len(pos) == len(prob)

In [8]:
Lbox = 1000.0

Nh = len(pos)
Nr = len(pos_rand)

### 1. Halo-Halo

In [9]:
k_hh = 8

tree_h = cKDTree(pos, boxsize=Lbox)
dist_hh, idx_hh = tree_h.query(pos, k=k_hh + 1)

# quitar self
dist_hh = dist_hh[:, 1:]
idx_hh  = idx_hh[:, 1:]

src_hh = np.repeat(np.arange(Nh), k_hh)
dst_hh = idx_hh.ravel()
d_hh = dist_hh.ravel()

### 2. Random-Random

In [10]:
k_rr = 8

tree_r = cKDTree(pos_rand, boxsize=Lbox)
dist_rr, idx_rr = tree_r.query(pos_rand, k=k_rr + 1)

dist_rr = dist_rr[:, 1:]
idx_rr = idx_rr[:, 1:]

src_rr = np.repeat(np.arange(Nr), k_rr)
dst_rr = idx_rr.ravel()
d_rr = dist_rr.ravel()

## Grafo

In [11]:
G = nx.Graph()

In [12]:
prob_arr = prob[['PVOID', 'PSHEET', 'PFILAMENT', 'PKNOT']].to_numpy()

In [13]:
Nh = len(pos)
Nr = len(pos_rand)
assert len(prob_arr) == Nh

In [14]:
for i in range(Nh):
    G.add_node(i, node_type='halo', pos=pos[i],
               PVOID=float(prob_arr[i, 0]),
               PSHEET=float(prob_arr[i, 1]),
               PFILAMENT=float(prob_arr[i, 2]),
               PKNOT=float(prob_arr[i, 3]))

In [15]:
for i in range(Nr):
    node_id = Nh + i
    G.add_node(node_id, node_type='random', pos=pos_rand[i],
               PVOID=1.0, PSHEET=0.0, PFILAMENT=0.0, PKNOT=0.0)

Creo que la idea seria algo como:

-> Halos = informacion

-> random voids = puntos donde reconstruir el campo DM

In [16]:
print('Nh =', Nh)
print('Nr =', Nr)

Nh = 309740
Nr = 130670


In [25]:
Lbox = 1000.0

### edges halos

In [26]:
k_hh = 8

In [18]:
tree_h = cKDTree(pos, boxsize=Lbox)

In [20]:
dist_hh, idx_hh = tree_h.query(pos, k=k_hh + 1)

# quitar self
dist_hh = dist_hh[:, 1:]
idx_hh = idx_hh[:, 1:]

src_hh = np.repeat(np.arange(Nh), k_hh)
dst_hh = idx_hh.ravel()
d_hh = dist_hh.ravel()

In [21]:
G.add_edges_from((int(i), int(j),
                  {'edge_type': 'HH', 'distance': float(d)})
                 for i, j, d in zip(src_hh, dst_hh, d_hh))

### edges rand

In [22]:
k_rr = 8

In [23]:
tree_r = cKDTree(pos_rand, boxsize=Lbox)

In [24]:
dist_rr, idx_rr = tree_r.query(pos_rand, k=k_rr + 1)

dist_rr = dist_rr[:, 1:]
idx_rr = idx_rr[:, 1:]

src_rr = np.repeat(np.arange(Nr), k_rr)
dst_rr = idx_rr.ravel()
d_rr = dist_rr.ravel()

In [27]:
G.add_edges_from((Nh + int(i), Nh + int(j),
                  {'edge_type': 'RR', 'distance': float(d)})
                 for i, j, d in zip(src_rr, dst_rr, d_rr))

----------------

In [28]:
print('Nodes:', G.number_of_nodes())
print('Edges:', G.number_of_edges())

Nodes: 440410
Edges: 2162502


In [31]:
print(Counter(data['node_type'] for _, data in G.nodes(data=True)))
print(Counter(data['edge_type'] for _, _, data in G.edges(data=True)))

Counter({'halo': 309740, 'random': 130670})
Counter({'HH': 1532295, 'RR': 630207})


halos                         random voids

●──●──●                       ○──○──○
│╲ │                          │ ╲  │
●─●──●         +              ○──○─○
   ╲                             ╲
    ●                             ○

In [32]:
G.nodes[0]

{'node_type': 'halo',
 'pos': array([686.011  , 533.36017, 426.34787], dtype=float32),
 'PVOID': 0.0,
 'PSHEET': 0.14000000059604645,
 'PFILAMENT': 0.6599999666213989,
 'PKNOT': 0.19999998807907104}

In [34]:
G.nodes[Nh+1]

{'node_type': 'random',
 'pos': array([668.1629 , 342.78842, 301.57013], dtype=float32),
 'PVOID': 1.0,
 'PSHEET': 0.0,
 'PFILAMENT': 0.0,
 'PKNOT': 0.0}